# Chapter 20 — Ensembles: Bagging, Random Forests, Boosting

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then `docs/TROUBLESHOOTING.md`.

In [1]:
!pip -q install -r https://raw.githubusercontent.com/FromAbsoluteZero/CodeBase/main/requirements.txt  # Colab only; skip locally

zsh:1: command not found: pip


## Create the data

Run once. Every dataset in this book is generated by code you can read — nothing is downloaded, so nothing can rot behind a dead link. This is the printed block from Chapter 14 (`code/ch14/gen_hr.py` in the repository).

In [2]:
import numpy as np, pandas as pd

rng = np.random.default_rng(11)
n = 1470
tenure   = np.clip(rng.gamma(2.2, 3.0, n), 0.2, 40).round(1)
salary   = np.clip(rng.normal(65000, 18000, n), 25000, 160000).round(-2)
overtime = (rng.random(n) < 0.28).astype(int)
commute  = np.clip(rng.gamma(2.0, 6.0, n), 1, 60).round(0)
satis    = np.clip(rng.normal(3.3, 0.95, n), 1, 5).round(1)
promo    = np.clip(rng.gamma(1.6, 1.6, n), 0, 15).round(1)
dept     = rng.choice(["Sales", "R&D", "Support"], n,
                      p=[0.32, 0.45, 0.23])

# the relationship the model will have to rediscover
z = (-2.55 + 1.15*overtime - 0.135*tenure - 0.60*(satis - 3.3)
     + 0.024*commute + 0.095*promo - 0.000014*(salary - 65000)
     + np.where(dept == "Sales", 0.45,
                np.where(dept == "Support", 0.20, 0.0)))
left = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)

hr = pd.DataFrame({"Department": dept, "YearsAtCompany": tenure,
    "MonthlyIncome": (salary/12).round(0),
    "OverTime": np.where(overtime == 1, "Yes", "No"),
    "CommuteMinutes": commute, "JobSatisfaction": satis,
    "YearsSincePromotion": promo, "Attrition": left})
hr.to_csv("hr.csv", index=False)
print(f"{len(hr):,} employees, "
      f"attrition rate {hr['Attrition'].mean():.1%}")

1,470 employees, attrition rate 12.2%


## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session. This cell is `code/ch20/_lib.py`.

In [3]:
import numpy as np, pandas as pd, warnings; warnings.filterwarnings("ignore")
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, BaggingClassifier,
                              GradientBoostingClassifier,
                              HistGradientBoostingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold)
from sklearn.metrics import roc_auc_score
# hr.csv is created by Chapter 14 (code/ch14/gen_hr.py). The blocks read it from the
# working directory exactly as the book does; if it is not here yet, use the copy shipped in
# data/generated/ (byte-identical to what the generator writes).
import os as _os, shutil as _shutil
if not _os.path.exists("hr.csv"):
    for _d in ("../../data/generated", "../data/generated", "data/generated"):
        if _os.path.exists(_os.path.join(_d, "hr.csv")):
            _shutil.copy(_os.path.join(_d, "hr.csv"), "hr.csv"); break
hr = pd.read_csv("hr.csv")
X = pd.get_dummies(hr.drop(columns="Attrition"),
                   columns=["Department", "OverTime"],
                   drop_first=True).astype(float)
y = hr["Attrition"].values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3,
                                      random_state=7, stratify=y)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
rng = np.random.default_rng(0)

## The chapter code

### Block 1  (`c1.py`)

In [4]:
# Why averaging helps at all, before any trees are involved. Each "model"
# is an unbiased but noisy estimate of the same truth. What matters is not
# how many you average but how much their errors have in common.
print(f"{'corr':>6}{'k=1':>9}{'k=10':>9}{'k=100':>9}{'reduction':>12}")
for corr in (0.0, 0.3, 0.6, 0.9):
    shared = rng.normal(0, np.sqrt(corr), 8000)
    own = rng.normal(0, np.sqrt(1 - corr), (8000, 100))
    row = []
    for k in (1, 10, 100):
        row.append((shared[:, None] + own[:, :k]).mean(1).var())
    print(f"{corr:>6.1f}{row[0]:>9.3f}{row[1]:>9.3f}{row[2]:>9.3f}"
          f"{row[0] / row[2]:>11.1f}x")

  corr      k=1     k=10    k=100   reduction
   0.0    1.020    0.101    0.010      101.1x
   0.3    0.976    0.368    0.302        3.2x
   0.6    0.983    0.635    0.599        1.6x
   0.9    1.014    0.925    0.918        1.1x


### Block 2  (`c2.py`)

In [5]:
# Bagging by hand: resample the rows with replacement, fit a tree on each,
# average the predicted probabilities.
def bag(n_trees, max_features=None, seed=0):
    r = np.random.default_rng(seed)
    preds = np.zeros((n_trees, len(Xte)))
    left_out = []
    for i in range(n_trees):
        idx = r.integers(0, len(Xtr), len(Xtr))       # bootstrap sample
        left_out.append(1 - len(np.unique(idx)) / len(Xtr))
        t = DecisionTreeClassifier(max_features=max_features,
                                   random_state=i)
        t.fit(Xtr.values[idx], ytr[idx])
        preds[i] = t.predict_proba(Xte.values)[:, 1]
    return preds, float(np.mean(left_out))

single = DecisionTreeClassifier(random_state=0).fit(Xtr, ytr)
p1 = single.predict_proba(Xte)[:, 1]
print(f"one unpruned tree        AUC {roc_auc_score(yte, p1):.4f}")
preds, oob = bag(200)
for k in (1, 5, 25, 200):
    print(f"bagged, {k:>3} trees        AUC "
          f"{roc_auc_score(yte, preds[:k].mean(0)):.4f}")
print(f"\neach bootstrap sample leaves out {oob:.1%} of the rows on average")
print(f"theory says 1/e = {1/np.e:.1%}   (these are the out-of-bag rows)")

one unpruned tree        AUC 0.5633


bagged,   1 trees        AUC 0.5011
bagged,   5 trees        AUC 0.5931
bagged,  25 trees        AUC 0.6177
bagged, 200 trees        AUC 0.6262

each bootstrap sample leaves out 36.8% of the rows on average
theory says 1/e = 36.8%   (these are the out-of-bag rows)


### Block 3  (`c3.py`)

In [6]:
# A random forest is bagging plus one more idea: at every split, consider
# only a random subset of features. That is what decorrelates the trees.
def bag(n_trees, max_features=None, seed=0):
    r = np.random.default_rng(seed)
    preds = np.zeros((n_trees, len(Xte)))
    for i in range(n_trees):
        idx = r.integers(0, len(Xtr), len(Xtr))
        t = DecisionTreeClassifier(max_features=max_features,
                                   random_state=i)
        t.fit(Xtr.values[idx], ytr[idx])
        preds[i] = t.predict_proba(Xte.values)[:, 1]
    return preds

print(f"{'features per split':>20}{'AUC':>9}{'mean pairwise corr':>21}")
for mf, label in [(None, "all 8"), (4, "4"), (3, "sqrt(8) ~ 3"), (2, "2")]:
    p = bag(200, max_features=mf)
    corr = np.corrcoef(p)[np.triu_indices(200, 1)].mean()
    print(f"{label:>20}{roc_auc_score(yte, p.mean(0)):>9.4f}{corr:>21.3f}")

  features per split      AUC   mean pairwise corr


               all 8   0.6262                0.165


                   4   0.6425                0.140


         sqrt(8) ~ 3   0.6372                0.135


                   2   0.6403                0.119


### Block 4  (`c4.py`)

In [7]:
# Boosting is not averaging. Each tree is fitted to what the ones before
# it got wrong, so the trees are deliberately dependent.
from sklearn.tree import DecisionTreeRegressor

f = np.zeros(len(Xtr))                    # running prediction, in log-odds
eta = 0.1
print(f"{'stage':>7}{'train log loss':>16}{'test AUC':>11}")
test_f = np.zeros(len(Xte))
for stage in range(1, 201):
    p = 1 / (1 + np.exp(-f))
    residual = ytr - p                     # negative gradient of log loss
    h = DecisionTreeRegressor(max_depth=3,
                              random_state=0).fit(Xtr, residual)
    f += eta * h.predict(Xtr)
    test_f += eta * h.predict(Xte)
    if stage in (1, 5, 25, 100, 200):
        pp = np.clip(1 / (1 + np.exp(-f)), 1e-9, 1 - 1e-9)
        ll = -np.mean(ytr*np.log(pp) + (1-ytr)*np.log(1-pp))
        print(f"{stage:>7}{ll:>16.4f}"
              f"{roc_auc_score(yte, test_f):>11.4f}")

  stage  train log loss   test AUC
      1          0.6777     0.6654
      5          0.6232     0.6648
     25          0.4617     0.6894


    100          0.3158     0.6957


    200          0.2730     0.7009


### Block 5  (`c5.py`)

In [8]:
cands = {
  "single tree (depth 4)": DecisionTreeClassifier(max_depth=4,
                                                 random_state=0),
  "bagged trees":          BaggingClassifier(n_estimators=300,
                                             random_state=0),
  "random forest":         RandomForestClassifier(n_estimators=300,
                                                  min_samples_leaf=5,
                                                  random_state=0),
  "gradient boosting":     GradientBoostingClassifier(n_estimators=200,
                                                      learning_rate=0.05,
                                                      max_depth=3,
                                                      random_state=0),
  "logistic regression":   make_pipeline(StandardScaler(),
                                         LogisticRegression(max_iter=1000)),
}
print(f"{'model':<24}{'CV AUC':>9}{'sd':>8}{'test AUC':>11}")
for name, m in cands.items():
    s = cross_val_score(m, Xtr, ytr, cv=cv, scoring="roc_auc")
    m.fit(Xtr, ytr)
    te = roc_auc_score(yte, m.predict_proba(Xte)[:, 1])
    print(f"{name:<24}{s.mean():>9.4f}{s.std():>8.4f}{te:>11.4f}")

model                      CV AUC      sd   test AUC


single tree (depth 4)      0.6645  0.0494     0.6489


bagged trees               0.6946  0.0576     0.6273


random forest              0.7319  0.0435     0.6716


gradient boosting          0.7284  0.0640     0.7014
logistic regression        0.7513  0.0463     0.7297


### Block 6  (`c6.py`)

In [9]:
# Chapter 19 claimed trees earn their advantage on interactions. Here is
# data with one: the outcome depends on whether two conditions AGREE,
# which no additive model can express.
r = np.random.default_rng(1)
n = 4000
a, b = r.normal(size=n), r.normal(size=n)
noise = r.normal(0, 1.1, n)
z = 1.1 * np.sign(a * b) + 0.4 * a + noise      # interaction dominates
yi = (z > 0).astype(int)
Xi = np.c_[a, b, r.normal(size=(n, 4))]        # plus four irrelevant columns

Xi_tr, Xi_te = Xi[:3000], Xi[3000:]
yi_tr, yi_te = yi[:3000], yi[3000:]

for name, m in [("logistic regression", make_pipeline(StandardScaler(),
                    LogisticRegression(max_iter=1000))),
                ("single tree (depth 4)", DecisionTreeClassifier(max_depth=4,
                    random_state=0)),
                ("random forest", RandomForestClassifier(n_estimators=300,
                    random_state=0)),
                ("gradient boosting", GradientBoostingClassifier(
                    n_estimators=200, learning_rate=0.05, random_state=0))]:
    m.fit(Xi_tr, yi_tr)
    pi = m.predict_proba(Xi_te)[:, 1]
    print(f"{name:<24}AUC {roc_auc_score(yi_te, pi):.4f}")

logistic regression     AUC 0.5950
single tree (depth 4)   AUC 0.8574


random forest           AUC 0.8603


gradient boosting       AUC 0.8716
